# Stage-1 Interpreter — FULL run (all 2761 LUTs) → Stage-13A eval

Fresh **A100 + High-RAM**, then set tokens in CELL 1 and Run All. Full 3-way interpreter:
caption every LUT, build the leakage-safe corpus, full-FT **Qwen2.5-0.5B-Instruct** (bf16 on A100),
and score the untouched `split_unit_id` holdout.

**Total ~2.5–4 h, dominated by captioning (CELL 2). Everything is resumable** — if the runtime
disconnects, just re-run the cell and it continues from its cache.

The slice run already cleared GO (routing/clarify/parse all solid). The headline to watch here is
**`attribute_f1[real_lut]`** — whether magnitude calibration climbs above the slice's ~0.10 with 5×
data — plus route accuracy, over-refusal, and per-route recalls holding up (Stage-13A gate,
docs/master_plan.md:317).

In [ ]:
# CELL 1 - provision (clone, install, tokens, stage corpus for active_rows.jsonl)
import os, pathlib, subprocess, sys
REPO, BRANCH = '/content/SLM', 'feat/two-stage'
if not pathlib.Path(REPO, '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/ericrcwu001/SLM', REPO], check=True)
os.chdir(REPO)
subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], check=True)
if not os.environ.get('SLM_PROVISIONED'):
    # [frontier] brings the openai SDK the teacher gateway needs; [sft] covers torch/transformers.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[sft,frontier]'], check=True)
    os.environ['SLM_PROVISIONED'] = '1'
print('HEAD:', subprocess.run(['git','log','--oneline','-1'], capture_output=True, text=True).stdout.strip())

def _env_token(name):
    for _p in ('/content/SLM/.env', '/content/.env', '.env'):
        fp = pathlib.Path(_p)
        if fp.is_file():
            for _l in fp.read_text().splitlines():
                if _l.strip().startswith(name + '='):
                    return _l.split('=', 1)[1].strip().strip('"').strip("'")
    return None

# ALWAYS re-prompt (Enter=keep the current value; type a new one to fix a wrong entry). Base URL
# is pre-filled.
import getpass
_DEFAULTS = {'TFY_BASE_URL': 'https://tfy.promptlens.trilogy.com/v1'}
for _n in ('HF_TOKEN', 'TFY_BASE_URL', 'TFY_API_KEY'):
    _cur = os.environ.get(_n) or _env_token(_n) or _DEFAULTS.get(_n, '')
    if not _cur:
        _hint = ' [required]'
    elif _n == 'TFY_BASE_URL':
        _hint = f' [Enter=keep {_cur}]'
    else:
        _hint = f' [Enter=keep ...{_cur[-4:]}]'
    os.environ[_n] = getpass.getpass(f'{_n}{_hint}: ').strip() or _cur
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')),
      '| TFY ready:', bool(os.environ.get('TFY_BASE_URL')) and bool(os.environ.get('TFY_API_KEY')))

os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
if not pathlib.Path('data/active_sft/active_rows.jsonl').is_file():
    subprocess.run(['slm_stage', 'stage', '--durable-root', 'hf://datasets/ericrcwu/LUT_SLM',
                    '--local-root', '/content/slm'], check=True)
print('active_rows.jsonl:', pathlib.Path('data/active_sft/active_rows.jsonl').is_file())

In [ ]:
# CELL 2 - FULL data: caption ALL 2761 LUTs + supplement (500/500) + unified corpus. ~1-2h, RESUMABLE.
import subprocess, sys, json
assert subprocess.run([sys.executable, '-m', 'scripts.generate_captions',
                       '--no-image', '--workers', '6']).returncode == 0, 'caption gen failed / handed off'
assert subprocess.run([sys.executable, '-m', 'scripts.generate_route_supplement']).returncode == 0, 'supplement failed'
assert subprocess.run([sys.executable, '-m', 'scripts.build_interpreter_corpus']).returncode == 0, 'corpus build failed'
print(json.dumps(json.load(open('data/interpreter/interpreter_corpus_manifest.json')), indent=2))

In [ ]:
# CELL 3 - train the interpreter on the FULL corpus (A100 bf16), run-id interp_full. ~30-60 min.
import subprocess, sys, glob
assert subprocess.run([sys.executable, '-m', 'interpreter.train', '--config',
                       'configs/candidate_interpreter.json', '--run-id', 'interp_full']).returncode == 0, 'train failed'
print('trained:', sorted(glob.glob('models/interpreter/interp_full_*'))[-1])

In [ ]:
# CELL 4 - score on the untouched holdout (output captured so the summary always prints).
# Full holdout is ~800+ rows, generated one-by-one -> ~15-30 min. For a fast preview add
# '--limit', '300' to the args below; drop it for the real Stage-13A number.
import subprocess, sys, glob
ADAPTER = sorted(glob.glob('models/interpreter/interp_full_*'))[-1]
r = subprocess.run([sys.executable, '-m', 'interpreter.score', '--config',
                    'configs/candidate_interpreter.json', '--adapter', ADAPTER],
                   capture_output=True, text=True)
print(r.stdout[-3000:])
print(r.stderr[-800:] if r.returncode else '')